# Bổ sung: Pseudo-label ABSA (keyword + sentiment)

Mục tiêu: tạo **`labeled_absa_auto.jsonl`** để Notebook **04** train ABSA (multi-label), **không** phụ thuộc module ngoài `Notebook_Report/`.

## Input / output
- **Input:** `absa_clean_reviews.csv` (Notebook 02).
- **Output:** `absa/absa_unlabeled.jsonl`, `absa/labeled_absa_auto.jsonl`, `absa/labeling_metadata.json`.

## Cách gán nhãn (đã nâng cấp)
- **Aspect:** keyword (6 nhóm) + **`overall`** (luôn có — sentiment theo **toàn bộ review**).
- **Chế độ mặc định — `sentence`:** tách câu → VADER (hoặc method bạn chọn) **trên từng câu** có keyword → **vote đa số** sentiment cho từng aspect. Tránh lỗi “một sentiment cho mọi aspect” như bản cũ.
- **Chế độ `document`:** giống pipeline cũ (một sentiment cho cả đoạn, gán cho mọi aspect phát hiện được) — đặt `ABSA_AUTOLABEL_MODE=document` nếu cần so sánh.

**Sentiment:** `vader` (mặc định, cần `vaderSentiment`), hoặc `textblob` / `sentiwordnet` / `fallback`. Đổi qua `ABSA_AUTOLABEL_SENTIMENT`.

Pseudo-label vẫn **có nhiễu**; mục tiêu là dữ liệu huấn luyện có cấu trúc ABSA hợp lý cho DistilRoBERTa ở notebook 04.


In [1]:
import json
import re
from pathlib import Path

import pandas as pd

# -----------------------------
# Cấu hình I/O trong Notebook_Report/
# -----------------------------
NOTEBOOK_DIR = Path.cwd()  # kỳ vọng bạn đang chạy notebook trong Notebook_Report/
# fallback: nếu mở notebook ở chỗ khác, vẫn cố gắng tìm đúng folder Notebook_Report
if NOTEBOOK_DIR.name != "Notebook_Report":
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "Notebook_Report" / "cinesense_reviews.csv").exists():
            NOTEBOOK_DIR = p / "Notebook_Report"
            break

ABSA_CLEAN_CSV = NOTEBOOK_DIR / "absa_clean_reviews.csv"
OUT_DIR = NOTEBOOK_DIR / "absa"
OUT_DIR.mkdir(parents=True, exist_ok=True)

UNLABELED_JSONL = OUT_DIR / "absa_unlabeled.jsonl"
LABELED_JSONL = OUT_DIR / "labeled_absa_auto.jsonl"

if not ABSA_CLEAN_CSV.exists():
    raise FileNotFoundError(
        f"Không thấy {ABSA_CLEAN_CSV}. Bạn chạy Notebook 02 để tạo file clean cho ABSA trước nhé."
    )

print("Notebook dir:", NOTEBOOK_DIR)
print("Input:", ABSA_CLEAN_CSV)
print("Output dir:", OUT_DIR)


Notebook dir: /Users/kotori/CineSen/Notebook_Report
Input: /Users/kotori/CineSen/Notebook_Report/absa_clean_reviews.csv
Output dir: /Users/kotori/CineSen/Notebook_Report/absa


In [2]:
# -----------------------------
# 1) Export unlabeled JSONL từ CSV đã clean (Notebook 02)
# -----------------------------
LIMIT_ROWS = None  # bám sát quy mô dữ liệu ~9k review của project

# Không xử lý text ở notebook này nữa: chỉ load dữ liệu đã clean
usecols = ["review_id", "tmdb_id", "cleaned_content"]
df = pd.read_csv(ABSA_CLEAN_CSV, usecols=usecols).fillna("")

# Chuẩn tên cột để các bước sau thống nhất dùng `text`
df = df.rename(columns={"cleaned_content": "text"})

# Bỏ dòng trống sau cleaning
df = df[df["text"].astype(str).str.strip().str.len() > 0]

if LIMIT_ROWS is not None:
    df = df.head(LIMIT_ROWS)

print("Rows for unlabeled:", len(df))

n_written = 0
with UNLABELED_JSONL.open("w", encoding="utf-8") as f:
    for _, r in df.iterrows():
        rec = {
            "id": str(r["review_id"]),
            "tmdb_id": int(r["tmdb_id"]),
            "text": str(r["text"]),
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        n_written += 1

print(f"Wrote unlabeled: {n_written} -> {UNLABELED_JSONL}")


Rows for unlabeled: 12027
Wrote unlabeled: 12027 -> /Users/kotori/CineSen/Notebook_Report/absa/absa_unlabeled.jsonl


In [3]:
import re
# -----------------------------
# 2) Auto-label: keyword aspect + sentiment (VADER / TextBlob / …)
# -----------------------------
import os
from collections import Counter

ASPECTS = ["script", "acting", "visuals", "music", "pacing", "direction", "overall"]
SENTIMENTS = ["negative", "neutral", "positive"]

# "sentence" (khuyến nghị): sentiment theo từng câu, vote đa số cho mỗi aspect — gần ABSA hơn.
# "document": một sentiment cho cả review (cách cũ, nhanh, nhiễu hơn).
LABELING_MODE = os.environ.get("ABSA_AUTOLABEL_MODE", "sentence").strip().lower()
if LABELING_MODE not in ("sentence", "document"):
    LABELING_MODE = "sentence"

ASPECT_KEYWORDS = {
    "script": [
        "script",
        "story",
        "plot",
        "writing",
        "written",
        "screenplay",
        "narrative",
        "storyline",
        "dialogue",
        "dialog",
        "twist",
        "cliche",
        "cliché",
    ],
    "acting": [
        "acting",
        "performance",
        "performances",
        "actor",
        "actress",
        "cast",
        "starring",
        "played",
        "portrayal",
        "character",
        "chemistry",
        "oscar",
        "voice",
    ],
    "visuals": [
        "visual",
        "visuals",
        "cgi",
        "cinematography",
        "cinematic",
        "effects",
        "special effects",
        "animation",
        "animated",
        "shot",
        "shots",
        "lighting",
        "aesthetic",
        "vfx",
    ],
    "music": [
        "music",
        "score",
        "soundtrack",
        "sound track",
        "song",
        "songs",
        "composer",
        "audio",
    ],
    "pacing": [
        "pacing",
        "pace",
        "slow",
        "fast",
        "drag",
        "dragged",
        "rushed",
        "length",
        "long",
        "short",
        "boring",
        "tedious",
        "tight",
        "flow",
        "runtime",
    ],
    "direction": [
        "direction",
        "director",
        "directed",
        "filmmaking",
        "film-making",
        "helmed",
        "directorial",
        "vision",
    ],
}


def _normalize_for_keyword(s: str) -> str:
    return re.sub(r"[^a-z\s]", " ", (s or "").lower())


def _keyword_hits(text: str) -> set[str]:
    """Chỉ 6 aspect nội dung (không gồm overall)."""
    normalized = _normalize_for_keyword(text)
    words = set(normalized.split())
    found: set[str] = set()
    for aspect, keywords in ASPECT_KEYWORDS.items():
        for kw in keywords:
            if kw in words or kw in normalized:
                found.add(aspect)
                break
    return found


def detect_aspects(text: str) -> set[str]:
    """Aspect từ keyword + luôn có overall (dùng cho chế độ document)."""
    return _keyword_hits(text) | {"overall"}


def split_sentences(text: str) -> list[str]:
    """Tách câu đơn giản (review tiếng Anh)."""
    t = (text or "").strip()
    if not t:
        return []
    parts = re.split(r"(?<=[.!?])\s+", t)
    out = [p.strip() for p in parts if p and len(p.strip()) >= 2]
    return out if out else [t]


def _fallback_sentiment(text: str) -> str:
    t = (text or "").lower()
    pos = (
        "great",
        "good",
        "amazing",
        "excellent",
        "love",
        "best",
        "brilliant",
        "stunning",
        "outstanding",
        "positive",
        "masterpiece",
        "gem",
    )
    neg = (
        "bad",
        "terrible",
        "weak",
        "boring",
        "worst",
        "awful",
        "poor",
        "disappointing",
        "negative",
        "waste",
        "do not recommend",
        "mediocre",
    )
    has_pos = sum(1 for w in pos if w in t)
    has_neg = sum(1 for w in neg if w in t)
    if has_pos > has_neg:
        return "positive"
    if has_neg > has_pos:
        return "negative"
    return "neutral"


# Chọn 1 trong: "vader" | "textblob" | "sentiwordnet" | "fallback"
SENTIMENT_METHOD = os.environ.get("ABSA_AUTOLABEL_SENTIMENT", "vader").strip().lower() or "vader"

# VADER: một analyzer cho cả pipeline (tránh khởi tạo lại mỗi câu)
_VADER_ANALYZER = None


def _get_vader():
    global _VADER_ANALYZER
    if _VADER_ANALYZER is None:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

        _VADER_ANALYZER = SentimentIntensityAnalyzer()
    return _VADER_ANALYZER


def _sentiment_from_polarity(p: float, neg_th: float = -0.05, pos_th: float = 0.05) -> str:
    if p <= neg_th:
        return "negative"
    if p >= pos_th:
        return "positive"
    return "neutral"


def _sentiment_vader(text: str) -> str:
    compound = _get_vader().polarity_scores(text)["compound"]
    return _sentiment_from_polarity(compound)


def _sentiment_textblob(text: str) -> str:
    from textblob import TextBlob

    polarity = float(TextBlob(text).sentiment.polarity)
    return _sentiment_from_polarity(polarity)


def _sentiment_sentiwordnet(text: str) -> str:
    import nltk
    from nltk import pos_tag, word_tokenize
    from nltk.corpus import sentiwordnet as swn
    from nltk.corpus import wordnet as wn

    for pkg in [
        ("punkt", "tokenizers/punkt"),
        ("averaged_perceptron_tagger", "taggers/averaged_perceptron_tagger"),
        ("wordnet", "corpora/wordnet"),
        ("sentiwordnet", "corpora/sentiwordnet"),
    ]:
        name, path = pkg
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(name, quiet=True)

    def _to_wn_pos(tag: str):
        if tag.startswith("J"):
            return wn.ADJ
        if tag.startswith("V"):
            return wn.VERB
        if tag.startswith("N"):
            return wn.NOUN
        if tag.startswith("R"):
            return wn.ADV
        return None

    tokens = word_tokenize((text or "").lower())
    tagged = pos_tag(tokens)

    score = 0.0
    hit = 0

    for w, t in tagged:
        wn_pos = _to_wn_pos(t)
        if wn_pos is None:
            continue
        synsets = wn.synsets(w, pos=wn_pos)
        if not synsets:
            continue
        syn = synsets[0]
        swn_syn = swn.senti_synset(syn.name())
        score += float(swn_syn.pos_score()) - float(swn_syn.neg_score())
        hit += 1

    if hit == 0:
        return "neutral"

    avg = score / hit
    return _sentiment_from_polarity(avg)


def get_sentiment(text: str) -> str:
    method = (SENTIMENT_METHOD or "fallback").strip().lower()

    try:
        if method == "vader":
            return _sentiment_vader(text)
        if method == "textblob":
            return _sentiment_textblob(text)
        if method == "sentiwordnet":
            return _sentiment_sentiwordnet(text)
        return _fallback_sentiment(text)
    except Exception:
        return _fallback_sentiment(text)


def _majority_sentiment(votes: list[str]) -> str:
    if not votes:
        return "neutral"
    c = Counter(votes)
    return c.most_common(1)[0][0]


def auto_label_record(rec: dict) -> dict:
    text = (rec.get("text") or "").strip()
    if not text:
        rec["labels"] = []
        return rec

    if LABELING_MODE == "document":
        sentiment = get_sentiment(text)
        if sentiment not in SENTIMENTS:
            sentiment = "neutral"
        aspects = detect_aspects(text)
        rec["labels"] = [{"aspect": a, "sentiment": sentiment} for a in sorted(aspects)]
        return rec

    # --- sentence-level: sentiment riêng cho từng câu có keyword aspect ---
    aspect_votes: dict[str, list[str]] = {a: [] for a in ASPECT_KEYWORDS}
    for sent in split_sentences(text):
        if len(sent) < 4:
            continue
        hit_aspects = _keyword_hits(sent)
        if not hit_aspects:
            continue
        s = get_sentiment(sent)
        if s not in SENTIMENTS:
            s = "neutral"
        for a in hit_aspects:
            aspect_votes[a].append(s)

    overall_s = get_sentiment(text)
    if overall_s not in SENTIMENTS:
        overall_s = "neutral"

    labels: list[dict] = []
    for a in ASPECTS:
        if a == "overall":
            continue
        votes = aspect_votes.get(a, [])
        if votes:
            labels.append({"aspect": a, "sentiment": _majority_sentiment(votes)})
    labels.append({"aspect": "overall", "sentiment": overall_s})

    have = {x["aspect"] for x in labels}
    for a in sorted(_keyword_hits(text)):
        if a not in have:
            labels.append({"aspect": a, "sentiment": overall_s})
            have.add(a)

    labels.sort(
        key=lambda x: (ASPECTS.index(x["aspect"]) if x["aspect"] in ASPECTS else 99, x["aspect"])
    )
    rec["labels"] = labels
    return rec


print(f"LABELING_MODE={LABELING_MODE!r} | SENTIMENT_METHOD={SENTIMENT_METHOD!r}")

n_labeled = 0
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

lines = UNLABELED_JSONL.read_text(encoding="utf-8").splitlines()
iter_lines = tqdm(lines, desc="Auto-label") if tqdm else lines

with LABELED_JSONL.open("w", encoding="utf-8") as fout:
    for line in iter_lines:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        auto_label_record(rec)
        fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
        n_labeled += 1

meta = {
    "labeling_mode": LABELING_MODE,
    "sentiment_method": SENTIMENT_METHOD,
    "n_records": n_labeled,
    "schema_aspects": ASPECTS,
    "schema_sentiments": SENTIMENTS,
    "output": str(LABELED_JSONL.name),
}
meta_path = OUT_DIR / "labeling_metadata.json"
with meta_path.open("w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"Wrote labeled: {n_labeled} -> {LABELED_JSONL}")
print(f"Metadata -> {meta_path}")


LABELING_MODE='sentence' | SENTIMENT_METHOD='vader'


Auto-label:   0%|          | 0/12027 [00:00<?, ?it/s]

Wrote labeled: 12027 -> /Users/kotori/CineSen/Notebook_Report/absa/labeled_absa_auto.jsonl
Metadata -> /Users/kotori/CineSen/Notebook_Report/absa/labeling_metadata.json


In [4]:
# -----------------------------
# 3) Thống kê nhanh (báo cáo + sanity check)
# -----------------------------
from collections import Counter
from itertools import islice

aspect_counter = Counter()
sent_counter = Counter()
labels_per_doc = []

STATS_LIMIT = None  # None = đọc hết

with LABELED_JSONL.open("r", encoding="utf-8") as f:
    it = f if STATS_LIMIT is None else islice(f, STATS_LIMIT)
    for line in it:
        rec = json.loads(line)
        labels = rec.get("labels", [])
        labels_per_doc.append(len(labels))
        for lab in labels:
            aspect_counter[lab.get("aspect")] += 1
            sent_counter[lab.get("sentiment")] += 1

print("Top aspects (số lần xuất hiện trong nhãn):")
for k, v in aspect_counter.most_common(12):
    print(f"  {k}: {v}")

print("\nSentiment counts (tất cả cặp aspect×sentiment):")
for k, v in sent_counter.most_common():
    print(f"  {k}: {v}")

if labels_per_doc:
    import statistics as stats
    print(f"\nNhãn / review: min={min(labels_per_doc)} max={max(labels_per_doc)} "
          f"mean={stats.mean(labels_per_doc):.2f}")


Top aspects (số lần xuất hiện trong nhãn):
  overall: 12027
  acting: 7888
  script: 7605
  pacing: 5443
  visuals: 4642
  direction: 2913
  music: 2404

Sentiment counts (tất cả cặp aspect×sentiment):
  positive: 27302
  neutral: 10704
  negative: 4916

Nhãn / review: min=1 max=7 mean=3.57
